In [2]:
import pandas as pd

# File paths
gene_expr_file = "/home/users/turbodu/kzlinlab/projects/morpho_integration/out/turbo/scala/exon_norm_paired_2000.csv"
feature_file = "/home/users/turbodu/kzlinlab/projects/morpho_integration/out/turbo/scala/m1_patchseq_morph_features.csv"
output_file = "/home/users/turbodu/kzlinlab/projects/morpho_integration/out/turbo/scala/feature_paired.csv"

# Step 1: Read gene expression data to get the cell IDs (in order)
print("Reading gene expression data...")
gene_expr_df = pd.read_csv(gene_expr_file)
reference_cell_ids = gene_expr_df['cell id'].tolist()
print(f"Found {len(reference_cell_ids)} cells in gene expression data")

# Step 2: Read feature file
print("Reading feature file...")
feature_df = pd.read_csv(feature_file)
print(f"Found {len(feature_df)} cells in feature file")

# Step 3: Filter and align feature data to match gene expression cell IDs
print("Filtering and aligning features...")
# Set 'cell id' as index for easy filtering
feature_df = feature_df.set_index('cell id')

# Extract only the cells that are in reference data, maintaining order
aligned_feature_df = feature_df.loc[reference_cell_ids]

# Reset index to make 'cell id' a column again
aligned_feature_df = aligned_feature_df.reset_index()

print(f"Extracted {len(aligned_feature_df)} cells")

# Step 4: Save the aligned feature data
print(f"Saving to {output_file}...")
aligned_feature_df.to_csv(output_file, index=False)

print("Done!")
print(f"Output shape: {aligned_feature_df.shape}")

Reading gene expression data...
Found 645 cells in gene expression data
Reading feature file...
Found 646 cells in feature file
Filtering and aligning features...
Extracted 645 cells
Saving to /home/users/turbodu/kzlinlab/projects/morpho_integration/out/turbo/scala/feature_paired.csv...
Done!
Output shape: (645, 64)


In [5]:
import pandas as pd

# Read the file
input_file = "/home/users/turbodu/kzlinlab/projects/morpho_integration/out/turbo/scala/feature_paired.csv"

print("Reading file...")
df = pd.read_csv(input_file)

print(f"\nShape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())

print(f"\nColumn names:")
print(df.columns.tolist())

print(f"\nData types:")
print(df.dtypes)

print(f"\nColumn types summary:")
print(df.dtypes.value_counts())

print(f"\nLast column name: '{df.columns[-1]}'")
print(f"Last column dtype: {df[df.columns[-1]].dtype}")

Reading file...

Shape: (645, 64)

First few rows:
             cell id  "apical" branch points  "apical" height  \
0  20171204_sample_2                    19.0           780.93   
1  20171204_sample_4                    20.0           696.13   
2  20171204_sample_5                    31.0           716.21   
3  20171204_sample_6                    41.0           607.64   
4  20171207_sample_2                    54.0           735.66   

   "apical" log1p number of outer bifurcations  \
0                                     2.302585   
1                                     0.693147   
2                                     2.079442   
3                                     2.197225   
4                                     3.218876   

   "apical" mean bifurcation distance  "apical" robust height  \
0                            0.383099               697.20675   
1                            0.195981               621.74275   
2                            0.253603               649.68275 

In [10]:
import pandas as pd
import numpy as np

def convert_fraction_to_float(value):
    """Convert fraction strings like '2/3' to float"""
    if pd.isna(value):
        return np.nan
    if isinstance(value, str):
        # Check if it's a fraction like '2/3'
        if '/' in value:
            try:
                parts = value.split('/')
                return float(parts[0]) / float(parts[1])
            except:
                return np.nan
        # Try to convert to float directly
        try:
            return float(value)
        except:
            return np.nan
    return float(value)

# File paths
input_file = "/home/users/turbodu/kzlinlab/projects/morpho_integration/out/turbo/scala/feature_paired.csv"
output_file = "/home/users/turbodu/kzlinlab/projects/morpho_integration/out/turbo/scala/feature_paired_normalized.csv"

# Step 1: Read the aligned feature data
print("Reading feature data...")
df = pd.read_csv(input_file)
print(f"Original shape: {df.shape}")

# Step 2: Drop the last column
print("Dropping last column...")
df = df.iloc[:, :-1]
print(f"Shape after dropping last column: {df.shape}")

# Step 3: Convert all columns (except 'cell id') to numeric, handling fractions
print("Converting fraction strings to floats...")
cell_id_col = 'cell id'

for col in df.columns:
    if col == cell_id_col:
        continue
    
    # Apply fraction conversion to the column
    df[col] = df[col].apply(convert_fraction_to_float)

print("Conversion complete!")

# Step 4: Normalize each numeric column
print("Normalizing columns...")

normalized_count = 0
for col in df.columns:
    if col == cell_id_col:
        continue
    
    # Get the column data
    col_data = df[col]
    
    # Record NA positions
    na_mask = col_data.isna()
    
    # Get non-NA values
    non_na_values = col_data[~na_mask]
    
    if len(non_na_values) > 0:
        # Calculate min and max of non-NA values
        min_val = non_na_values.min()
        max_val = non_na_values.max()
        
        # Normalize: (x - min) / (max - min) + 1
        if max_val != min_val:  # Avoid division by zero
            normalized = (non_na_values - min_val) / (max_val - min_val) + 1
        else:
            # If all non-NA values are the same, set them to 1.5 (middle of [1, 2])
            normalized = pd.Series([1.5] * len(non_na_values), index=non_na_values.index)
        
        # Update the column with normalized values
        df.loc[~na_mask, col] = normalized
    
    # Set all NA values to 0
    df.loc[na_mask, col] = 0
    
    normalized_count += 1
    if normalized_count % 100 == 0:
        print(f"  Processed {normalized_count}/{len(df.columns)-1} columns...")

print(f"Normalization complete! Processed {normalized_count} columns.")

# Step 5: Save the normalized data
print(f"\nSaving to {output_file}...")
df.to_csv(output_file, index=False)

print("Done!")
print(f"Final shape: {df.shape}")
print(f"Output saved to: {output_file}")

Reading feature data...
Original shape: (645, 64)
Dropping last column...
Shape after dropping last column: (645, 63)
Converting fraction strings to floats...
Conversion complete!
Normalizing columns...
Normalization complete! Processed 62 columns.

Saving to /home/users/turbodu/kzlinlab/projects/morpho_integration/out/turbo/scala/feature_paired_normalized.csv...
Done!
Final shape: (645, 63)
Output saved to: /home/users/turbodu/kzlinlab/projects/morpho_integration/out/turbo/scala/feature_paired_normalized.csv


In [8]:
#!/usr/bin/env python
"""
Simple diagnostic for m1_patchseq_morph_features.csv
"""
import os
import pandas as pd

file_path = "/home/users/turbodu/kzlinlab/projects/morpho_integration/out/turbo/scala/m1_patchseq_morph_features.csv"

print("=" * 80)
print("CELL CLASS FILE DIAGNOSTIC")
print("=" * 80)

# Check file exists
if not os.path.exists(file_path):
    print("ERROR: File not found!")
    exit(1)

print("File:", file_path)
print("File size:", os.path.getsize(file_path) / 1024, "KB")

# Read first few lines
print("\n" + "-" * 80)
print("First 3 lines (raw):")
print("-" * 80)

with open(file_path, 'r') as f:
    for i in range(3):
        line = f.readline()
        print("\nLine", i, ":")
        print("  Length:", len(line), "characters")
        print("  First 150 chars:", line[:150])
        print("  Commas:", line.count(','))
        print("  Tabs:", line.count('\t'))
        print("  Quotes:", line.count('"'))

# Try different parsing methods
print("\n" + "-" * 80)
print("Testing pandas parsing methods:")
print("-" * 80)

methods = [
    ("Method 1: Standard comma", {"sep": ","}),
    ("Method 2: Tab", {"sep": "\t"}),
    ("Method 3: Auto-detect", {"sep": None, "engine": "python"}),
]

for name, kwargs in methods:
    print("\n" + name)
    try:
        df = pd.read_csv(file_path, nrows=2, **kwargs)
        print("  SUCCESS! Shape:", df.shape)
        print("  Number of columns:", len(df.columns))
        print("  First 3 columns:", df.columns.tolist()[:3])
        print("  Last 3 columns:", df.columns.tolist()[-3:])
        
        # Check for required columns
        has_cell_id = 'cell id' in df.columns
        has_cell_class = 'cell class' in df.columns
        
        if has_cell_id:
            print("  >>> FOUND 'cell id' column!")
        else:
            print("  >>> 'cell id' NOT found, searching...")
            for col in df.columns:
                if 'cell' in col.lower() and 'id' in col.lower():
                    print("      Found similar:", col)
                    break
        
        if has_cell_class:
            print("  >>> FOUND 'cell class' column!")
        else:
            print("  >>> 'cell class' NOT found, searching...")
            for col in df.columns:
                if 'class' in col.lower():
                    print("      Found similar:", col)
                    break
        
    except Exception as e:
        print("  FAILED:", str(e)[:100])

print("\n" + "=" * 80)
print("DIAGNOSTIC COMPLETE")
print("=" * 80)

CELL CLASS FILE DIAGNOSTIC
File: /home/users/turbodu/kzlinlab/projects/morpho_integration/out/turbo/scala/m1_patchseq_morph_features.csv
File size: 405.134765625 KB

--------------------------------------------------------------------------------
First 3 lines (raw):
--------------------------------------------------------------------------------

Line 0 :
  Length: 1496 characters
  First 150 chars: cell id,"""apical"" branch points","""apical"" height","""apical"" log1p number of outer bifurcations","""apical"" mean bifurcation distance","""apica
  Commas: 63
  Tabs: 0
  Quotes: 54

Line 1 :
  Length: 585 characters
  First 150 chars: 20180306_sample_2,22.0,347.16,2.19722457733622,0.2035580499465356,281.13950000000006,241.16650000000004,0.12028344688745655,2903.7322249937706,354.690
  Commas: 63
  Tabs: 0
  Quotes: 0

Line 2 :
  Length: 473 characters
  First 150 chars: 20190418_sample_4,19.0,250.38,1.0986122886681098,0.2176818811439505,208.884,104.7975,0.1351810401602212,1620.061644

In [9]:
import pandas as pd
import numpy as np
import scanpy as sc
import os

sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=80, facecolor='white')

print("=" * 80)
print("DIFFERENTIAL EXPRESSION ANALYSIS USING SCANPY")
print("=" * 80)

# ============================================================================
# STEP 1: Load gene expression data
# ============================================================================
print("\n[1] Loading gene expression data...")
gene_expr_path = "/home/users/turbodu/kzlinlab/projects/morpho_integration/out/turbo/scala/exon_norm_paired_2000.csv"
gene_expr_df = pd.read_csv(gene_expr_path)

if gene_expr_df.columns[0] in ['Unnamed: 0', 'cell id', 'Cell', '']:
    gene_expr_df = gene_expr_df.set_index(gene_expr_df.columns[0])

print(f"   Gene expression shape: {gene_expr_df.shape}")
print(f"   Cells: {gene_expr_df.shape[0]}, Genes: {gene_expr_df.shape[1]}")

# ============================================================================
# STEP 2: Load RNA family metadata (TAB-SEPARATED)
# ============================================================================
print("\n[2] Loading RNA family metadata...")
rna_family_path = "/home/users/turbodu/kzlinlab/projects/morpho_integration/out/turbo/scala/m1_patchseq_meta_data.csv"
rna_family_df = pd.read_csv(rna_family_path, sep='\t')
print(f"   ✓ Loaded: {rna_family_df.shape}")

# ============================================================================
# STEP 3: Load cell class metadata (COMMA with QUOTES issue)
# ============================================================================
print("\n[3] Loading cell class metadata...")
cell_class_path = "/home/users/turbodu/kzlinlab/projects/morpho_integration/out/turbo/scala/m1_patchseq_morph_features.csv"

# The file has problematic quotes in column names
# We'll read it and clean the column names
print("   Handling quote issues in column names...")

# Read the file
cell_class_df = pd.read_csv(cell_class_path)

# Clean column names - remove triple quotes and extra quotes
print(f"   Original columns: {len(cell_class_df.columns)}")
print(f"   First column: '{cell_class_df.columns[0]}'")
print(f"   Last column: '{cell_class_df.columns[-1]}'")

# Clean the column names
cleaned_columns = []
for col in cell_class_df.columns:
    # Remove leading/trailing quotes and spaces
    cleaned = col.strip().strip('"').strip("'")
    # Replace """ with nothing
    cleaned = cleaned.replace('"""', '')
    # Replace "" with "
    cleaned = cleaned.replace('""', '"')
    cleaned_columns.append(cleaned)

cell_class_df.columns = cleaned_columns

print(f"   After cleaning:")
print(f"   First column: '{cell_class_df.columns[0]}'")
print(f"   Last column: '{cell_class_df.columns[-1]}'")

# Verify we have the required columns
if 'cell id' not in cell_class_df.columns:
    print(f"\n   ERROR: 'cell id' still not found after cleaning!")
    print(f"   All columns: {cell_class_df.columns.tolist()}")
    
    # Try to find it
    for i, col in enumerate(cell_class_df.columns):
        if 'cell' in col.lower() and 'id' in col.lower():
            print(f"   Found at position {i}: '{col}'")
            cell_class_df = cell_class_df.rename(columns={col: 'cell id'})
            break
    else:
        # Use first column
        print(f"   Using first column as 'cell id'")
        cell_class_df = cell_class_df.rename(columns={cell_class_df.columns[0]: 'cell id'})

if 'cell class' not in cell_class_df.columns:
    print(f"\n   ERROR: 'cell class' not found after cleaning!")
    
    # Try to find it
    for i, col in enumerate(cell_class_df.columns):
        if 'class' in col.lower():
            print(f"   Found at position {i}: '{col}'")
            cell_class_df = cell_class_df.rename(columns={col: 'cell class'})
            break
    else:
        # Use last column
        print(f"   Using last column as 'cell class'")
        cell_class_df = cell_class_df.rename(columns={cell_class_df.columns[-1]: 'cell class'})

print(f"   ✓ Final shape: {cell_class_df.shape}")
print(f"   ✓ 'cell id' column: {('cell id' in cell_class_df.columns)}")
print(f"   ✓ 'cell class' column: {('cell class' in cell_class_df.columns)}")

# ============================================================================
# STEP 4: Align RNA family data
# ============================================================================
print("\n[4] Aligning RNA family data...")

rna_family_df_indexed = rna_family_df.set_index('Cell')
common_cells_rna = gene_expr_df.index.intersection(rna_family_df_indexed.index)
print(f"   Common cells: {len(common_cells_rna)}")

gene_expr_rna = gene_expr_df.loc[common_cells_rna]
rna_family_aligned = rna_family_df_indexed.loc[common_cells_rna]

# Remove NA labels
rna_family_aligned = rna_family_aligned[rna_family_aligned['RNA family'].notna()]
gene_expr_rna = gene_expr_rna.loc[rna_family_aligned.index]
print(f"   Final cells: {len(rna_family_aligned)}")

print(f"\n   RNA family distribution:")
print(rna_family_aligned['RNA family'].value_counts())

# ============================================================================
# STEP 5: Align cell class data
# ============================================================================
print("\n[5] Aligning cell class data...")

cell_class_df_indexed = cell_class_df.set_index('cell id')
common_cells_class = gene_expr_df.index.intersection(cell_class_df_indexed.index)
print(f"   Common cells: {len(common_cells_class)}")

gene_expr_class = gene_expr_df.loc[common_cells_class]
cell_class_aligned = cell_class_df_indexed.loc[common_cells_class]

# Remove NA labels
cell_class_aligned = cell_class_aligned[cell_class_aligned['cell class'].notna()]
gene_expr_class = gene_expr_class.loc[cell_class_aligned.index]
print(f"   Final cells: {len(cell_class_aligned)}")

print(f"\n   Cell class distribution:")
print(cell_class_aligned['cell class'].value_counts())

# ============================================================================
# STEP 6: Create AnnData objects
# ============================================================================
print("\n[6] Creating AnnData objects...")

adata_rna = sc.AnnData(
    X=gene_expr_rna.values.astype(np.float32),
    obs=rna_family_aligned,
    var=pd.DataFrame(index=gene_expr_rna.columns)
)
adata_rna.obs_names = gene_expr_rna.index
adata_rna.var_names = gene_expr_rna.columns
print(f"   RNA family AnnData: {adata_rna.shape}")

adata_class = sc.AnnData(
    X=gene_expr_class.values.astype(np.float32),
    obs=cell_class_aligned,
    var=pd.DataFrame(index=gene_expr_class.columns)
)
adata_class.obs_names = gene_expr_class.index
adata_class.var_names = gene_expr_class.columns
print(f"   Cell class AnnData: {adata_class.shape}")

# ============================================================================
# STEP 7: Check data transformation
# ============================================================================
print("\n[7] Checking data...")
max_val = gene_expr_rna.values.max()
print(f"   Data range: [{gene_expr_rna.values.min():.2f}, {max_val:.2f}]")

if max_val > 20:
    print("   Applying log transformation...")
    adata_rna.X = np.log1p(adata_rna.X)
    adata_class.X = np.log1p(adata_class.X)
else:
    print("   Data already log-transformed")

# ============================================================================
# STEP 8-9: Run DE analysis
# ============================================================================
print("\n[8] Running DE analysis for RNA family...")
sc.tl.rank_genes_groups(
    adata_rna,
    groupby='RNA family',
    method='wilcoxon',
    use_raw=False,
    corr_method='benjamini-hochberg',
    pts=True,
    key_added='rank_genes_rna_family'
)
print("   ✓ Completed")

print("\n[9] Running DE analysis for cell class...")
sc.tl.rank_genes_groups(
    adata_class,
    groupby='cell class',
    method='wilcoxon',
    use_raw=False,
    corr_method='benjamini-hochberg',
    pts=True,
    key_added='rank_genes_cell_class'
)
print("   ✓ Completed")

# ============================================================================
# STEP 10: Save results
# ============================================================================
print("\n[10] Saving results...")
output_dir = "/home/users/turbodu/kzlinlab/projects/morpho_integration/out/turbo/scala/DE_results"
os.makedirs(output_dir, exist_ok=True)

# RNA family results
groups_rna = adata_rna.uns['rank_genes_rna_family']['names'].dtype.names
results_rna_list = []
for group in groups_rna:
    group_df = pd.DataFrame({
        'gene_name': adata_rna.uns['rank_genes_rna_family']['names'][group],
        'cluster_name': group,
        'p_value': adata_rna.uns['rank_genes_rna_family']['pvals'][group],
        'p_value_adj': adata_rna.uns['rank_genes_rna_family']['pvals_adj'][group],
        'log_fold_change': adata_rna.uns['rank_genes_rna_family']['logfoldchanges'][group],
        'score': adata_rna.uns['rank_genes_rna_family']['scores'][group]
    })
    results_rna_list.append(group_df)

results_rna_df = pd.concat(results_rna_list, ignore_index=True)
results_rna_df.to_csv(os.path.join(output_dir, "DE_results_RNA_family.csv"), index=False)
results_rna_df.groupby('cluster_name').head(50).to_csv(
    os.path.join(output_dir, "DE_results_RNA_family_top50.csv"), index=False
)
print(f"   ✓ RNA family results saved")

# Cell class results
groups_class = adata_class.uns['rank_genes_cell_class']['names'].dtype.names
results_class_list = []
for group in groups_class:
    group_df = pd.DataFrame({
        'gene_name': adata_class.uns['rank_genes_cell_class']['names'][group],
        'cluster_name': group,
        'p_value': adata_class.uns['rank_genes_cell_class']['pvals'][group],
        'p_value_adj': adata_class.uns['rank_genes_cell_class']['pvals_adj'][group],
        'log_fold_change': adata_class.uns['rank_genes_cell_class']['logfoldchanges'][group],
        'score': adata_class.uns['rank_genes_cell_class']['scores'][group]
    })
    results_class_list.append(group_df)

results_class_df = pd.concat(results_class_list, ignore_index=True)
results_class_df.to_csv(os.path.join(output_dir, "DE_results_cell_class.csv"), index=False)
results_class_df.groupby('cluster_name').head(50).to_csv(
    os.path.join(output_dir, "DE_results_cell_class_top50.csv"), index=False
)
print(f"   ✓ Cell class results saved")

# ============================================================================
# Summary
# ============================================================================
print("\n" + "=" * 80)
print("ANALYSIS COMPLETE!")
print("=" * 80)
print(f"\n✓ RNA Family Analysis:")
print(f"  - Groups: {len(adata_rna.obs['RNA family'].unique())}")
print(f"  - Cells: {adata_rna.n_obs}")
print(f"  - Genes: {adata_rna.n_vars}")

print(f"\n✓ Cell Class Analysis:")
print(f"  - Groups: {len(adata_class.obs['cell class'].unique())}")
print(f"  - Cells: {adata_class.n_obs}")
print(f"  - Genes: {adata_class.n_vars}")

print(f"\n📁 Results saved to: {output_dir}")
print("\nGenerated files:")
print("  - DE_results_RNA_family.csv")
print("  - DE_results_RNA_family_top50.csv")
print("  - DE_results_cell_class.csv")
print("  - DE_results_cell_class_top50.csv")
print("\n" + "=" * 80)

DIFFERENTIAL EXPRESSION ANALYSIS USING SCANPY

[1] Loading gene expression data...
   Gene expression shape: (645, 2000)
   Cells: 645, Genes: 2000

[2] Loading RNA family metadata...
   ✓ Loaded: (1329, 33)

[3] Loading cell class metadata...
   Handling quote issues in column names...
   Original columns: 64
   First column: 'cell id'
   Last column: 'cell class'
   After cleaning:
   First column: 'cell id'
   Last column: 'cell class'
   ✓ Final shape: (646, 64)
   ✓ 'cell id' column: True
   ✓ 'cell class' column: True

[4] Aligning RNA family data...
   Common cells: 645
   Final cells: 645

   RNA family distribution:
RNA family
IT             154
Pvalb          145
Sst            108
CT              78
Vip             63
Lamp5           47
ET              33
low quality      8
Sncg             6
NP               3
Name: count, dtype: int64

[5] Aligning cell class data...
   Common cells: 645
   Final cells: 645

   Cell class distribution:
cell class
inh    371
exc    274
Name